In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

import os
os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')

In [2]:
#retrieve data
cps14 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps14_with_nearest_college_imgfeat.parquet')
# cps13 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps13_with_nearest_college_imgfeat_pca.parquet')
# cps12 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps12_with_nearest_college_imgfeat_pca.parquet')

cps14.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place',
       ...
       'mosaiks_3990', 'mosaiks_3991', 'mosaiks_3992', 'mosaiks_3993',
       'mosaiks_3994', 'mosaiks_3995', 'mosaiks_3996', 'mosaiks_3997',
       'mosaiks_3998', 'mosaiks_3999'],
      dtype='object', length=4024)

In [3]:
cps14.head()

,person_num,log_wage,educ,exp,exp2,female,black,mv,age,birth_place,...,mosaiks_3990,mosaiks_3991,mosaiks_3992,mosaiks_3993,mosaiks_3994,mosaiks_3995,mosaiks_3996,mosaiks_3997,mosaiks_3998,mosaiks_3999
0,4,16.659324,0.0,20.0,400.0,1,0,7.0,26,1,...,0.054907,3.583759,1.316642,0.716429,1.452152,1.471457,0.116492,5.285796,0.007212,5.026694
1,3,16.979841,12.0,8.0,64.0,0,1,5.0,26,1,...,0.007007,3.190889,1.198570,0.456456,1.538894,1.547372,0.012397,5.090301,0.002686,4.947719
2,2,15.806328,12.0,0.0,0.0,0,1,5.0,18,1,...,0.000149,3.201558,1.322542,0.202977,1.509819,1.623803,0.007932,5.479442,0.023653,5.330654
3,1,16.964838,14.0,23.0,529.0,0,0,7.0,43,1,...,0.005186,3.919045,1.438792,0.710858,1.448475,1.489014,0.010588,5.751905,0.017108,5.456117
4,2,17.585665,18.0,17.0,289.0,1,0,7.0,41,1,...,0.005186,3.919045,1.438792,0.710858,1.448475,1.489014,0.010588,5.751905,0.017108,5.456117


In [15]:
print(cps14["nearest_college_dist_km"].isna().mean())

0.008866034818847885


In [4]:
#dropna
cps14 = cps14.dropna(subset=["nearest_college_dist_km"]).copy()

In [5]:
#baseline 2SLS

formula1 = "log_wage ~ 1 + exp + exp2 + [educ ~ nearest_college_dist_km]"
base = IV2SLS.from_formula(formula=formula1, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

formula2 = "log_wage ~ 1 + exp + exp2 + female + black + [educ ~ nearest_college_dist_km]"
base2 = IV2SLS.from_formula(formula=formula2, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

print(base.summary)
print(base.first_stage)
print(base2.summary)
print(base2.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                      0.0250
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0250
No. Observations:              147451   F-statistic:                    5110.5
Date:                Sat, Apr 04 2026   P-value (F-stat)                0.0000
Time:                        03:31:15   Distribution:                  chi2(3)
Cov. Estimator:             clustered                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      13.948     0.1803     77.371     0.0000      13.594      14.301
exp            0.0236     0.0041     5.8072     0.00

In [6]:
# TSLS (baseline) with PUMA-fixed effect
formula3 = "log_wage ~ 1 + exp + exp2 + C(PUMA) + [educ ~ nearest_college_dist_km]"
base3 = IV2SLS.from_formula(formula=formula3, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

formula4 = "log_wage ~ 1 + exp + exp2 + female + black + C(PUMA) + [educ ~ nearest_college_dist_km]"
base4 = IV2SLS.from_formula(formula=formula4, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

print(base3.summary)
print(base3.first_stage)
print(base4.summary)
print(base4.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                      0.1217
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1148
No. Observations:              147451   F-statistic:                 6.993e+16
Date:                Mon, Apr 06 2026   P-value (F-stat)                0.0000
Time:                        14:54:33   Distribution:               chi2(1152)
Cov. Estimator:             clustered                                         
                                                                              
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------
Intercept            14.473     0.5336     27.122     0.0000      13.427      15.519
exp                  0.0348 

In [3]:
# Two-stage Least Square with img feat
# img feat全部ぶちこみだと同じ列があるからエラー！Double selection or PCA必須！

DATA = cps14.copy()
IMG_COLS = [c for c in DATA.columns if c.startswith('mosaiks')]

if not IMG_COLS:
    raise ValueError('No PCA image columns found. Expected columns starting with mosaiks_.')

# PUMA-level regressors are perfectly collinear with PUMA fixed effects, so do not include C(puma).
base_cols = ['log_wage', 'educ', 'nearest_college_dist_km', 'exp', 'exp2', 'female', 'black', "pums_weight"]
model_data = DATA[base_cols + IMG_COLS].dropna().copy()

img_terms = ' + '.join(IMG_COLS)
formula = (
    'log_wage ~ 1 + exp + exp2 + female + black + '
    f'+ {img_terms} + [educ ~ nearest_college_dist_km]'
)

res = IV2SLS.from_formula(formula=formula, data=model_data, weights=model_data["pums_weight"]).fit(
    cov_type="clustered",
    clusters=model_data['PUMA_STATE']
)

print('n_obs:', len(model_data))
print('mosaiks_:', len(IMG_COLS))
print('formula:', formula)
print(res.summary)
print(res.first_stage)


ValueError: regressors [exog endog] do not have full column rank

: 

In [11]:
base_cols = ['log_wage', 'educ', 'nearest_college_dist_km', 'exp', 'exp2', 'female', 'black', 'age_bins', 'puma']
model_data = DATA[base_cols + IMG_COLS].dropna().copy()

img_terms = ' + '.join(IMG_COLS)
formula = (
    'log_wage ~ 1 + exp + exp2 + female + black + [educ ~ nearest_college_dist_km]'
)

res = IV2SLS.from_formula(formula=formula, data=model_data).fit(
    cov_type="clustered",
    clusters=model_data['puma']
)

print('n_obs:', len(model_data))
print('n_img_pca:', len(IMG_COLS))
print('formula:', formula)
print(res.summary)
print(res.first_stage)

n_obs: 1045
n_img_pca: 16
formula: log_wage ~ 1 + exp + exp2 + female + black + [educ ~ nearest_college_dist_km]
                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                     -0.0826
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0878
No. Observations:                1045   F-statistic:                    63.804
Date:                Thu, Mar 05 2026   P-value (F-stat)                0.0000
Time:                        15:16:31   Distribution:                  chi2(5)
Cov. Estimator:             clustered                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept     -0.3

In [12]:
base_cols = ['log_wage', 'educ', 'nearest_college_dist_km', 'exp', 'exp2', 'female', 'black', 'age_bins', 'puma']
model_data = DATA[base_cols + IMG_COLS].dropna().copy()

img_terms = ' + '.join(IMG_COLS)
formula = (
    'log_wage ~ 1 + [educ ~ nearest_college_dist_km]'
)

res = IV2SLS.from_formula(formula=formula, data=model_data).fit(
    cov_type="clustered",
    clusters=model_data['puma']
)

print('n_obs:', len(model_data))
print('n_img_pca:', len(IMG_COLS))
print('formula:', formula)
print(res.summary)
print(res.first_stage)

n_obs: 1045
n_img_pca: 16
formula: log_wage ~ 1 + [educ ~ nearest_college_dist_km]
                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                      0.0371
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0362
No. Observations:                1045   F-statistic:                    0.0403
Date:                Thu, Mar 05 2026   P-value (F-stat)                0.8409
Time:                        15:20:32   Distribution:                  chi2(1)
Cov. Estimator:             clustered                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      1.9543     3.0600     0.6387     